# ScanDar on Colab

Training moves between a local RTX 3060 and Colab depending on what is available, so nothing in the codebase assumes either one. This notebook makes a Colab runtime look like the local machine:

1. check what GPU we were given
2. mount Drive, so **checkpoints survive a session timeout**
3. clone the repo and install it
4. point `SCANDAR_DATA` / `SCANDAR_OUT` at Drive
5. run the sanity checks

After that, every other notebook and every `python train.py ...` command runs unchanged.

> **One-time setup:** copy `data/` to `MyDrive/scandar/data` (it is only ~30 MB before backgrounds).

In [ ]:
# 1. what did we get?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. Drive — checkpoints and figures live here so a timeout costs nothing
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/scandar'

In [ ]:
# 3. code
REPO_URL = 'https://github.com/SepehrGhr/ScanDar.git'

import os
if not os.path.isdir('/content/ScanDar'):
    !git clone $REPO_URL /content/ScanDar
%cd /content/ScanDar
!git pull --ff-only

# Colab already ships torch + CUDA, so requirements.txt deliberately does not pin them.
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
# 4. point the project at Drive — this is the whole portability story
import os
os.environ['SCANDAR_DATA'] = f'{DRIVE_ROOT}/data'
os.environ['SCANDAR_OUT'] = f'{DRIVE_ROOT}/outputs'

import scandar
print('data   ->', scandar.paths.data)
print('output ->', scandar.paths.out)

In [ ]:
# 5. verify before spending GPU time on a broken setup
!python scripts/prepare_data.py
!python scripts/sanity_checks.py

## Training here

Batch size and gradient accumulation are separate config keys precisely so the *effective* batch stays the same on a 6 GB laptop GPU and on whatever Colab hands out. A T4 or L4 can afford to trade accumulation for a bigger real batch:

```bash
python train.py --config configs/enhance.yaml --set train.batch_size=32 train.grad_accum=1
```

Every run checkpoints into `SCANDAR_OUT` with its optimiser and RNG state, so a session that dies mid-epoch resumes with `train.resume=auto` (the default) rather than starting over.